In [ ]:
import numpy as np
import json
import os
from crimm.IO.PDBParser import PDBParser
from crimm.Utils.StructureUtils import get_coords
from crimm.Data.ptable import PERIODIC_TABLE
from crimm.Modeller.TopoLoader import TopologyGenerator
from crimm.Modeller.TopoFixer import fix_chain
from tqdm.notebook import tqdm
from rdkit import Chem
from rdkit.Chem import AllChem
from scipy.spatial import KDTree
import pickle

In [ ]:
def generate_pocket(protein, lig_mol, pocket_cutoff):
    prot_atoms = [a for a in protein.get_atoms() if a.element != 'H']
    recep_coords = np.array([a.coord for a in prot_atoms])
    lig_coords = lig_mol.GetConformer(0).GetPositions()
    tree = KDTree(recep_coords)

    nei_ids = tree.query_ball_point(lig_coords, pocket_cutoff)
    
    _atom_ids = []
    for cur_list in nei_ids:
        _atom_ids += cur_list
    
    pocket_atom_ids = np.unique(_atom_ids)
    pocket_coords = recep_coords[pocket_atom_ids]
    pocket_atoms = np.array(list(protein.get_atoms()))[pocket_atom_ids]
    
    z_pocket = np.array(
        [PERIODIC_TABLE[a.element.capitalize()]['number'] for a in pocket_atoms]
    )
    z_lig = np.array([a.GetAtomicNum() for a in lig_mol.GetAtoms()])
    # 0: receptor pocket, 1: ligand 
    labels = np.concatenate([np.zeros_like(z_pocket), np.ones_like(z_lig)])
    z = np.concatenate([z_pocket, z_lig])
    pos = np.concatenate([pocket_coords, lig_coords])
    return z, pos, labels

def sep_entities(structure):
    chains = structure.models[0].chains
    if len(chains) > 2:
        return None, None
    for chain in structure.models[0]:
        if chain.chain_type == "Polypeptide(L)":
            protein = chain
        elif chain.chain_type == "Heterogens":
            if len(chain) > 1:
                return None, None
            ligand = chain.residues[0]
        else:
            return None, None
    return protein, ligand

def get_elements(atoms):
    elements = []
    for a in atoms:
        element = a.element.capitalize()
        if element not in PERIODIC_TABLE:
            return None
        elements.append(PERIODIC_TABLE[element]['number'])
    return np.array(elements)

def generate_prot_h(protein, lig_mol, pocket_cutoff):
    if protein is None or lig_mol is None:
        return None
    topo = TopologyGenerator()
    topo.generate(protein)
    hs = fix_chain(protein)

    recep_coords = get_coords(protein)
    lig_coords = lig_mol.GetConformer(0).GetPositions()
    tree = KDTree(recep_coords)

    nei_ids = tree.query_ball_point(lig_coords, pocket_cutoff)

    _atom_ids = []
    for cur_list in nei_ids:
        _atom_ids+=cur_list
    
    pocket_atom_ids = np.unique(_atom_ids)
    n_atoms = pocket_atom_ids.shape[0]
    if n_atoms > 200 or n_atoms < 30:
        return None
    pocket_coords = recep_coords[pocket_atom_ids]
    pocket_atoms = np.array(list(protein.get_atoms()))[pocket_atom_ids]
    z_pocket = get_elements(pocket_atoms)
    z_lig = np.array([a.GetAtomicNum() for a in lig_mol.GetAtoms()])
    return z_lig, lig_coords, z_pocket, pocket_coords

In [ ]:
def generate_sep_pocket(protein, lig_mol, pocket_cutoff):
    prot_atoms = [a for a in protein.get_atoms() if a.element != 'H']
    recep_coords = np.array([a.coord for a in prot_atoms])
    lig_coords = lig_mol.GetConformer(0).GetPositions()
    tree = KDTree(recep_coords)

    nei_ids = tree.query_ball_point(lig_coords, pocket_cutoff)
    
    _atom_ids = []
    for cur_list in nei_ids:
        _atom_ids += cur_list
    
    pocket_atom_ids = np.unique(_atom_ids)
    pocket_coords = recep_coords[pocket_atom_ids]
    pocket_atoms = np.array(list(protein.get_atoms()))[pocket_atom_ids]
    
    z_pocket = np.array(
        [PERIODIC_TABLE[a.element.capitalize()]['number'] for a in pocket_atoms]
    )
    z_lig = np.array([a.GetAtomicNum() for a in lig_mol.GetAtoms()])
    return z_lig, lig_coords, z_pocket, pocket_coords

In [ ]:
work_dir = '/path/to/diffdock_pocket_output/'  # raw LIT-PCBA docked poses (external)
structure_dir = work_dir + 'furyal_docked_poses/'
with open(work_dir+'target_lig_ids.json', 'r') as f:
    lig_ids = json.load(f)

In [ ]:
with open(os.path.join(work_dir, 'lit-pcba-labels.pkl'), 'rb') as f:
    labels = pickle.load(f)

In [ ]:
lig_ids['ADRB2'].keys()

In [ ]:
import sys
sys.setrecursionlimit(2000)

In [ ]:
pocket_cutoff = 4.0
parser = PDBParser(QUIET=True)
protein_vecs = {}
missing = []
failed = []
for target, pdbids in tqdm(lig_ids.items(), total=len(lig_ids)):
    cur_dict = protein_vecs[target] = {}
    for pdbid in pdbids:
        if len(lig_ids[target][pdbid]) == 0:
            missing.append((target, pdbid))
            continue
        lig_id = lig_ids[target][pdbid][0]
        pdb_dir = os.path.join(
            structure_dir, f'{target.upper()}_complexes', pdbid
        )
        lig_path = os.path.join(pdb_dir, f'{lig_id}_ligand.sdf')
        recept_path = os.path.join(pdb_dir, f'{lig_id}_receptor.pdb')
        if not (os.path.exists(lig_path) and os.path.exists(recept_path)):
            missing.append((target, pdbid, lig_id))
            continue
        structure = parser.get_structure(recept_path, pdbid)
        model = structure[0]
        protein = model['A']
        protein.can_seq = protein.seq
        protein.reported_res = [(i, res.resname) for i, res in enumerate(protein, start=1)]
        if not protein.is_continuous():
            failed.append((target, pdbid, lig_id))
            continue
        lig_mol = Chem.MolFromMolFile(lig_path)
        lig_mol_h = AllChem.AddHs(lig_mol, addCoords=True)
        try:
            results = generate_prot_h(protein, lig_mol_h, pocket_cutoff=pocket_cutoff)
        except:
            failed.append((target, pdbid, lig_id))
            continue
        if results is None:
            failed.append((target, pdbid, lig_id))
            continue
        z_lig, lig_coords, z_pocket, pocket_coords = results
        cur_dict[pdbid] = (z_pocket, pocket_coords)

In [ ]:
failed

In [ ]:
empty_targets = []
for target, pdbids in protein_vecs.items():
    if len(pdbids) == 0:
        empty_targets.append(target)

In [ ]:
for target in empty_targets:
    protein_vecs.pop(target)

In [ ]:
with open(f'./h_pocket_4.pkl', 'rb') as f:
    protein_vecs = pickle.load(f)

In [ ]:
# pocket_cutoff = 4.0
# parser = PDBParser(QUIET=True)
ligand_vecs = {}
target_iter = tqdm(protein_vecs.items(), total=len(protein_vecs))
missing = []
special_cases = []
for target, pdbids in target_iter:
    if target in ligand_vecs:
        continue
    target_iter.set_description_str(f"Processing {target}")
    vecs = {}
    pdbid = list(pdbids.keys())[0]
    cur_lig_ids = lig_ids[target][pdbid]
    pdb_dir = os.path.join(
        structure_dir, f'{target.upper()}_complexes', pdbid
    )
    for lig_id in tqdm(cur_lig_ids):
        lig_path = os.path.join(pdb_dir, f'{lig_id}_ligand.sdf')
        recept_path = os.path.join(pdb_dir, f'{lig_id}_receptor.pdb')
        if not (os.path.exists(lig_path) and os.path.exists(recept_path)):
            missing.append((target, pdbid, lig_id))
            continue
        lig_mol = Chem.MolFromMolFile(lig_path)
        if lig_mol is None:
            print(f'Ligand parsing failed {pdbid}-{lig_id}')
            continue
        lig_mol_h = AllChem.AddHs(lig_mol, addCoords=True)
        lig_coords = lig_mol_h.GetConformer(0).GetPositions()
        z_lig = np.array([a.GetAtomicNum() for a in lig_mol_h.GetAtoms()])
        label = labels.get((target, pdbid, lig_id), None)
        if label is None:
            print(f"{target}, {pdbid}, {lig_id} label missing")
            continue
        if len(z_lig) > 80:
            special_cases.append((target, pdbid, lig_id))
            continue
        if len(z_lig) < 6:
            special_cases.append((target, pdbid, lig_id))
            continue
        vecs[(pdbid, lig_id)] = (
            z_lig,
            lig_coords,
            label
        )
    ligand_vecs[target] = vecs
    # break

In [ ]:
with open(f'./h_pocket_4.pkl', 'wb') as f:
    pickle.dump(protein_vecs, f)


In [ ]:
with open(f'./h_ligands.pkl', 'wb') as f:
    pickle.dump(ligand_vecs, f)

In [ ]:
with open(f'./h_pocket_4.pkl', 'rb') as f:
    protein_vecs = pickle.load(f)
with open(f'./h_ligands.pkl', 'rb') as f:
    ligand_vecs = pickle.load(f)

In [ ]:
ligand_vecs.keys()

In [ ]:
lig_mol_h

In [ ]:
target_vecs.keys()

In [ ]:
target_vecs.pop('ESR1_ago')
target_vecs.pop('ESR1_ant')

In [ ]:
with open(f'./h_pcba_sep_pocket_vecs_4.pkl', 'wb') as f:
    pickle.dump(target_vecs, f)

In [ ]:
lig_ids['ADRB2'].keys()

In [ ]:
labels

In [ ]:
len(labels)

In [ ]:
lig_ids.keys()